# Silver — conform and inspect one reject

Origami Health · synthetic training only. Website: lab1. Planned block: 30 minutes.

Run selected prepared cells; do not provision services in class. Configure the full asset pack and runtime bindings before running. Live AIDP execution and classroom timing remain unverified.


## First-time user guide

**Learning objective:** Separate valid, invalid and duplicate claims, then prove that no source record vanished.

**Before you begin:** Notebook 01 finished successfully for your assigned slot. Keep the original synthetic source files unchanged.

**Key terms:** Silver is the conformed data layer. A business key identifies a claim. Deduplication keeps the first source_record_id for each claim_id. Quarantine keeps invalid records with their reasons; it is not an insurance claim denial.

**Timebox:** This is the shared 30-minute block for notebooks 01 AND 02, not 30 minutes each. Read the explanations as you run the prepared cells. Optional exploration is not part of the core timebox.

**How to run:** There are 2 code cells below. Run them one at a time, from top to bottom, using the editor's run-cell control. Wait for completion, read the actual output and only then continue. Do not use Run all on your first pass. All runtime bindings are facilitator-prepared; do not paste credentials or edit another participant's paths.

For common problems, open START_HERE.md at the asset-pack root. If an assertion fails, stop and keep the error for the facilitator; do not change the assertion or expected values to make it pass.


## Load administrator-prepared participant bindings

**Purpose:** Load administrator-prepared participant bindings

**Inputs:** Full asset pack plus external ORIGAMI_ASSET_ROOT / ORIGAMI_RUNTIME_CONFIG bindings.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Assigned slot and synthetic snapshot; placeholders fail closed.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 1:** Run the binding cell and confirm your slot.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
import os, sys, json
from pathlib import Path
asset_root = os.environ.get('ORIGAMI_ASSET_ROOT')
if not asset_root:
    raise RuntimeError('Facilitator must configure ORIGAMI_ASSET_ROOT and ORIGAMI_RUNTIME_CONFIG before class.')
sys.path.insert(0, str(Path(asset_root) / 'shared'))
from origami_core import AS_OF, SOURCES, fingerprint, score_rows
from aidp_runtime import configuration, bounded_rows, persist, read_delta, source_frame, verify_fixture, unique, publish_snapshot, save_evidence
cfg = configuration()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sparkContext.addPyFile(str(Path(asset_root) / 'shared' / 'origami_core.py'))
print('Participant:', cfg['participant_id'], '| synthetic snapshot:', AS_OF)

## Evaluate, deduplicate, quarantine and reconcile

**Purpose:** Evaluate, deduplicate, quarantine and reconcile

**Inputs:** Persisted Bronze tables, explicit source schemas and deterministic DQ rules.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** 725 input = 720 accepted + 4 rejected + 1 duplicate; inspect R0722.

**Business meaning:** A rejected record is a data-quality issue, not a denied claim or fraud finding.

**Your action — step 2:** Before running the transformation, predict what 725 = 720 + 4 + 1 means. Run the cell, inspect the reject table and find R0722. Read its reject_reason instead of editing the source to remove the error.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
from origami_core import cents, valid_date
from pyspark.sql.types import LongType, BooleanType
money=F.udf(cents,LongType()); date_ok=F.udf(valid_date,BooleanType())
source_columns=json.loads((Path(asset_root)/'contracts'/'expected.json').read_text())['source_columns']
refs={}
ref_keys={'members':['member_id'],'plans':['plan_id','plan_version'],'enrollments':['enrollment_id'],'providers':['provider_id'],'benefit_schedule':['benefit_id'],'benefit_usage':['usage_id']}
for name in SOURCES:
    if name!='claims':
        refs[name]=persist(spark,cfg,'silver_'+name,read_delta(spark,cfg,'bronze_'+name).select(*source_columns[name]),ref_keys[name])
raw=read_delta(spark,cfg,'bronze_claims').select(*source_columns['claims'])
ranked=raw.withColumn('_rank',F.row_number().over(Window.partitionBy('claim_id').orderBy('source_record_id')))
duplicates=ranked.where('_rank > 1').drop('_rank').withColumn('reject_reason',F.lit('DUPLICATE_CLAIM'))
base=ranked.where('_rank = 1').drop('_rank')
members=refs['members'].select(F.col('member_id').alias('_member'))
providers=refs['providers'].select(F.col('provider_id').alias('_provider'))
enroll=refs['enrollments'].select(F.col('enrollment_id').alias('_enrollment'),F.col('member_id').alias('_enrolled_member'),F.col('start_date').alias('_start'),F.col('end_date').alias('_end'))
evaluated=(base.join(members,base.member_id==members._member,'left')
    .join(providers,base.provider_id==providers._provider,'left')
    .join(enroll,base.enrollment_id==enroll._enrollment,'left')
    .withColumn('_amount',money('submitted_amount')).withColumn('_paid',money('paid_amount')))
evaluated=evaluated.withColumn('reject_reason',F.concat_ws('|',
    F.when(F.col('_member').isNull(),'MEMBER_REFERENCE'),
    F.when(F.col('_provider').isNull(),'PROVIDER_REFERENCE'),
    F.when(F.col('_amount').isNull() | F.col('_paid').isNull() | (F.col('_paid')>F.col('_amount')),'AMOUNT'),
    F.when(~date_ok('service_date'),'SERVICE_DATE'),
    F.when(F.col('_enrollment').isNull() | (F.col('_enrolled_member')!=F.col('member_id')) | (F.col('service_date')<F.col('_start')) | (F.col('service_date')>F.col('_end')),'ENROLLMENT'),
    F.when(F.col('currency')!='PHP','CURRENCY'))).cache()
try:
    rejects=evaluated.where("reject_reason != ''").select(*source_columns['claims'],'reject_reason')
    accepted=evaluated.where("reject_reason = ''").select('claim_id','member_id','enrollment_id','provider_id','service_code','service_date',F.col('_amount').alias('submitted_centavos'),F.col('_paid').alias('paid_centavos'),'currency','status',F.col('processing_days').cast('long').alias('processing_days'),F.col('missing_attachment').cast('long').alias('missing_attachment'),F.col('prior_duplicate_signal').cast('long').alias('prior_duplicate_signal'),'feature_available_date','source_record_id')
    for name,frame,key in [('silver_claims',accepted,'claim_id'),('silver_rejects',rejects,'source_record_id'),('silver_duplicates',duplicates,'source_record_id')]:
        saved=persist(spark,cfg,name,frame,[key]);verify_fixture(spark,cfg,name,saved)
    assert raw.count()==accepted.count()+rejects.count()+duplicates.count()
    rejects.select('source_record_id','claim_id','reject_reason').show(truncate=False)
    save_evidence(cfg,'silver',{'input':raw.count(),'accepted':accepted.count(),'rejected':rejects.count(),'duplicates':duplicates.count()})
finally:
    evaluated.unpersist()

## Completion checkpoint

- [ ] Every required code cell completed without error.
- [ ] I checked the actual result: There are 720 accepted claims, 4 rejected source records and 1 duplicate. Their source IDs are disjoint and cover all 725 inputs. The saved fixtures match the reference fingerprints.
- [ ] I saved evidence: Record all four counts and R0722’s actual reason text. Monetary values are now integer centavos: 100 centavos = PHP 1. The notebook saves silver.json.

**Discuss:** What is the difference between a duplicate, a data-quality reject and a fraud finding?

**Self-check explanation:** A duplicate repeats a business record; a reject fails a data rule. Neither establishes fraud. Fraud investigation requires additional evidence and human review.

**Next:** day1/03_gold.ipynb. Continue only after the core checkpoint passes, or after the facilitator explicitly records a labelled fallback.

## If you get stuck

Note the notebook name, cell heading and sanitized error. Missing configuration, permission errors and missing target tables are facilitator setup issues, not instructions to provision resources. Unexpected values are validation failures: do not erase tables, disable checks or rerun the entire pipeline blindly. See START_HERE.md for the troubleshooting table.
